In [ ]:
#HIERARCHY
# Page
#  ├── Blocks
#  │    ├── Lines
#  │    │    ├── Spans
#  │    │    │    └── TEXT
#  │    │    └── bbox
#  │    └── bbox
#  └── Images

# 🧠 PyMuPDF Structure (Transposed Table)

| Attribute              | **Level 1: Page**      | **Level 2: Block**        | **Level 3: Line**        | **Level 4: Span**       |
| ---------------------- | ---------------------- | ------------------------- | ------------------------ | ----------------------- |
| **What it represents** | Whole page             | Paragraph/section chunk   | One row of text          | Smallest styled text    |
| **Core container**     | `blocks[]`, `images[]` | `lines[]`                 | `spans[]`                | actual text             |
| **Main content key**   | —                      | —                         | —                        | `text` ⭐                |
| **Index/order**        | `pgn` (page no.)       | `number`                  | —                        | —                       |
| **Type info**          | —                      | `type` (0 = text)         | `wmode` (0 = horizontal) | `flags`, `char_flags`   |
| **Position**           | —                      | `bbox`                    | `bbox`                   | `bbox`, `origin`        |
| **Direction**          | —                      | —                         | `dir` (1,0 = L→R)        | —                       |
| **Style info**         | —                      | —                         | —                        | `size`, `font`, `color` |
| **Hierarchy role**     | Top level              | Groups lines              | Groups spans             | Holds actual text       |
| **Notes**              | Entry point            | Not always true paragraph | May split oddly          | Splits on style change  |

---

## 📏 bbox (applies to Block / Line / Span)

* `(x0, y0, x1, y1)`
* Origin = top-left
* Y increases downward

---

## 🧠 Mental Model

**Page → Block → Line → Span → Text**

---


# 🧠 PyMuPDF Structure (Compact Cheat Sheet)

```text
PAGE
 ├── pgn → page number
 ├── blocks[]
 │    ├── number → block index
 │    ├── type → 0 = text
 │    ├── bbox → (x0, y0, x1, y1)
 │    ├── lines[]
 │    │    ├── bbox → line box
 │    │    ├── dir → text direction (1,0)
 │    │    ├── spans[]
 │    │    │    ├── text → actual text ⭐
 │    │    │    ├── size → font size
 │    │    │    ├── font → font name
 │    │    │    ├── color → int color
 │    │    │    └── bbox → span box
 │
 └── images[] → image data
```

---

## 📏 bbox reminder

```text
(x0, y0, x1, y1)
top-left origin, y goes ↓
```

---

## ⚠️ mental model

```text
Page → Blocks → Lines → Spans → Text
```

---


# 📄 PyMuPDF Text Extraction Structure

---
## 🧱 LEVEL 1 — PAGE
```python
{
  "pgn": int,
  "blocks": list,
  "images": list
}
```
### 🔑 Keys
* **`pgn`** → Page number (0-based)
* **`blocks`** → List of text/image blocks on the page
* **`images`** → List of images on the page

---
## 📦 LEVEL 2 — BLOCK
```python
{
  "number": int,
  "type": int,
  "bbox": (x0, y0, x1, y1),
  "lines": list
}
```
### 🔑 Keys
* **`number`** → Block index/order on page
* **`type`** → Block type

  * `0 = text`
  * other values = image/graphics
* **`bbox`** → Bounding box of block
* **`lines`** → List of text lines

### 📌 Notes

* A block ≈ paragraph / section
* Not always perfect (PDFs are chaotic)

---
## 📄 LEVEL 3 — LINE
```python
{
  "spans": list,
  "wmode": int,
  "dir": (x, y),
  "bbox": (x0, y0, x1, y1)
}
```
### 🔑 Keys
* **`spans`** → List of styled text pieces
* **`wmode`** → Writing mode (`0 = horizontal`)
* **`dir`** → Text direction vector

  * `(1, 0)` → left → right
* **`bbox`** → Line bounding box

### 📌 Notes

* A line = one row of text
* Lines may break oddly due to layout

---
## ✍️ LEVEL 4 — SPAN (TEXT LEVEL)
```python
{
  "text": str,
  "size": float,
  "font": str,
  "color": int,
  "flags": int,
  "char_flags": int,
  "ascender": float,
  "descender": float,
  "origin": (x, y),
  "bbox": (x0, y0, x1, y1)
}
```
### 🔑 Keys
* **`text`** → Actual text content ⭐
* **`size`** → Font size
* **`font`** → Font name
* **`color`** → Text color (int encoded)
* **`flags`** → Style flags (bold/italic etc.)
* **`char_flags`** → Character-level flags
* **`ascender / descender`** → Font metrics
* **`origin`** → Starting position
* **`bbox`** → Bounding box

### 📌 Notes

* A span = smallest styled text unit
* New span = font/style change

---
## 📏 BOUNDING BOX (bbox)
```python
(x0, y0, x1, y1)
```
* `x0` → left
* `y0` → top
* `x1` → right
* `y1` → bottom

### 📌 Coordinate System

* Origin = top-left
* Y increases downward

## ⚠️ IMPORTANT GOTCHAS

* Blocks ≠ always paragraphs
* Lines may split unexpectedly
* Spans split on style changes
* PDF layout ≠ logical reading order

---
